# 📸 League of Legends Background Screenshot Collector

Capture clean gameplay background frames for synthetic dataset composition.

> ⚠️ **IMPORTANT: Capture WITHOUT health bars or GUI!**
> - **In Replay Mode:** Press `U` to hide interface/timeline, and turn off **Show Health Bars** & **Show Summoner Names** in `Esc` -> `Interface` settings.
> - **In Practice Tool:** Press `Ctrl + Shift + H` to hide HUD and turn off health bars in settings.
>
> Capturing terrain free of native health bars ensures the synthetic generator produces unambiguous, perfectly labeled YOLO training data.


In [ ]:
!pip install mss pynput

In [ ]:
import time
import os
from mss import mss, tools
from pynput import keyboard

# --- CONFIGURATION ---
SAVE_FOLDER = os.path.join(os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd(), "Screenshots")
INTERVAL = 0.25

os.makedirs(SAVE_FOLDER, exist_ok=True)

is_running = True
wait_time = 10
print(f"Waiting for {wait_time} seconds before starting continuous capture...")
time.sleep(wait_time)

def on_press(key):
    global is_running
    if key == keyboard.Key.esc:
        print("\nStop key detected. Ending capture...")
        is_running = False
        return False

listener = keyboard.Listener(on_press=on_press)
listener.start()

print(f"Starting continuous capture. Saving to: {SAVE_FOLDER}")
print("Press 'Esc' at any time to stop.")

with mss() as sct:
    monitor = sct.monitors[1] 
    screenshot_count = 0
    
    while is_running:
        start_time = time.time()
        
        filename = f"replay_cap_{int(time.time())}_{screenshot_count:04d}.png"
        filepath = os.path.join(SAVE_FOLDER, filename)
        
        sct_img = sct.grab(monitor)
        
        with open(filepath, "wb") as output_file:
             # Notice it is now tools.to_png instead of mss.tools.to_png
             output_file.write(tools.to_png(sct_img.rgb, sct_img.size))
             
        screenshot_count += 1
        
        elapsed = time.time() - start_time
        sleep_time = INTERVAL - elapsed
        
        if sleep_time > 0 and is_running:
            time.sleep(sleep_time)

print(f"Capture finished. {screenshot_count} screenshots saved.")

In [ ]:

import os
import time
from mss import mss, tools

# --- CONFIGURATION ---
SAVE_FOLDER = os.path.join(os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd(), "Screenshots")
os.makedirs(SAVE_FOLDER, exist_ok=True)
wait_time = 5
print(f"Waiting for {wait_time} seconds before taking a single screenshot...")
time.sleep(wait_time)
with mss() as sct:
    monitor = sct.monitors[1]
    
    filename = f"single_shot_{int(time.time())}.png"
    filepath = os.path.join(SAVE_FOLDER, filename)
    
    sct_img = sct.grab(monitor)
    
    with open(filepath, "wb") as output_file:
         # Notice it is now tools.to_png instead of mss.tools.to_png
         output_file.write(tools.to_png(sct_img.rgb, sct_img.size))

print(f"Single screenshot saved to: {filepath}")